# SSC-MILP: one problem, live

Ten steps from problem text to verdict. Every stage runs now: the generator and the
requirement extractor call the DeepSeek API, GLPK solves the relaxations, SCIP solves the
integer programs. Nothing is read from `runs/`.

Change `BENCH` and `PID` in step 1 to run any problem of any benchmark.

## 1. Pick the problem

`BENCH` is one of `nl4opt`, `mamo_easy`, `mamo_complex`, `complexor`, `industryor`,
`optibench`, `optmath`. `PID` is a problem id of that benchmark; set it to `None` for the
first problem.

In [ ]:
import sys, os, json, time
sys.path.insert(0, os.path.abspath('.'))

from ssc import config as C
from ssc.data import load_benchmark

BENCH = 'nl4opt'
PID   = 'nl4opt_0001'

problems = load_benchmark(BENCH)
problem  = next(p for p in problems if PID is None or p['pid'] == PID)
PID      = problem['pid']
GOLD     = problem['answer']

print('benchmark :', BENCH, '  problems available :', len(problems))
print('problem   :', PID)
print('gold      :', GOLD)
print()
print(problem['question'])

## 2. Retrieve templates

Keyword scoring over the archetype library, Eq. 24. The retrieved skeletons go into the
generation prompt; the generic template is the fallback.

In [ ]:
from ssc.templates import retrieve_templates, template_score, TEMPLATES

scored = sorted(((template_score(t, problem['question']), t['name']) for t in TEMPLATES[:-1]),
                reverse=True)
print('top archetype scores:')
for s, n in scored[:5]:
    print('  %6.2f  %s' % (s, n))

templates = retrieve_templates(problem['question'])
print()
print('retrieved :', [t['name'] for t in templates])

## 3. Generate the pool, live

Five formulation variants times two repeats, temperatures 0.6 to 1.4, up to two repair
rounds at temperature 0. One chat completion per candidate, plus one per repair round.
Responses are cached on disk, so re-running this cell costs nothing.

In [ ]:
from ssc.llm import LLMClient
from ssc.generate import generate_pool

llm = LLMClient(cache_dir=f'runs/_live/cache/llm')
print('model :', llm.model)

t0   = time.time()
pool = generate_pool(llm, problem, workers=4)
print('generated %d candidates in %.1f s' % (len(pool['candidates']), time.time()-t0))
print('usage :', llm.usage)
print()
print('%-4s %-8s %-6s %-8s %-7s %s' % ('pos','variant','T','repeats','clean','defects at generation'))
for c in pool['candidates']:
    r = c['recipe']
    print('%-4d v%-7d %-6.1f %-8d %-7s %s' % (
        c['position'], r['variant'], r['temperature'], len(c['history'])-1,
        'yes' if not c['defects_at_generation'] else 'no', c['defects_at_generation']))

## 4. Read one candidate

The generator is constrained to a single `build_model()` function returning a Pyomo
`ConcreteModel`, with no solver call inside.

In [ ]:
codes = [c['code'] for c in pool['candidates']]
print(codes[0])

## 5. Parse into signatures

Each module is executed in an isolated subprocess, then the constructed model is reduced
to the comparable structure M = (V, o, R).

In [ ]:
from ssc.sandbox import parse_candidate
from ssc.signature import build_signature

parsed = [parse_candidate(c) for c in codes]
sigs   = [build_signature(p) for p in parsed]

s = sigs[0]
print('candidate 0  constructed =', s.constructed, '  defects =', parsed[0]['defects'])
for v in s.vars:
    print('  var %-14s domain=%-10s arity=%d bounds=%s' % (v.name, v.domain, v.arity, v.bounds))
print('  objective sense =', s.obj.sense if s.obj else None)
for r in s.rows[:5]:
    print('  row %-16s sense=%-3s terms=%d rhs=%s' % (r.comp, r.sense, r.n_terms, r.rhs))
print()
print('%-4s %-6s %-6s %-8s %s' % ('pos','vars','rows','has obj','defects'))
for i,(p,g) in enumerate(zip(parsed, sigs)):
    print('%-4d %-6d %-6d %-8s %s' % (i, len(g.vars), len(g.rows), g.has_obj, p['defects']))

## 6. SQ, static quality

Eq. 13: one minus the summed defect penalties, floored at zero. A fatal defect also caps
the symbolic base later.

In [ ]:
from ssc.quality import static_quality, has_fatal

SQ    = [static_quality(p['defects']) for p in parsed]
fatal = [has_fatal(p['defects'])      for p in parsed]
for i in range(len(SQ)):
    print('pos %d  SQ=%.3f  fatal=%-6s %s' % (i, SQ[i], fatal[i], parsed[i]['defects']))

## 7. TA, text alignment, live

One API call extracts weighted requirements over the closed 21-token vocabulary of
Appendix L. Eq. 14 then scores each candidate by the share of requirement weight its
signature covers.

In [ ]:
from ssc.extract import extract_requirements
from ssc.alignment import text_alignment, evidence_features

ex   = extract_requirements(llm, problem)
reqs = ex['requirements']
print('extraction ok =', ex['ok'], '  attempts =', len(ex['attempts']))
print('usage :', llm.usage)
print()
for r in reqs:
    print('  weight %.1f  %-30s %s' % (r['weight'], r['name'], r['features']))

TA = [text_alignment(g, reqs) for g in sigs]
print()
print('evidence features of candidate 0 :', sorted(evidence_features(sigs[0])))
print('TA :', [round(x,3) for x in TA])

## 8. SC, clone-aware structural consistency

Pairwise signature similarity (Eqs. 17 to 22), union-find clone grouping at 0.95, then a
credibility-weighted peer mean in which each clone group contributes once (Eqs. 11, 12, 23).

In [ ]:
from ssc.similarity import similarity_matrix, program_sim
from ssc.selector import clone_groups, structural_consistency

S      = similarity_matrix(sigs)
groups = clone_groups(S)
SC     = structural_consistency(S, SQ, TA, groups)

print('similarity matrix')
print('     ' + ''.join('%6d' % j for j in range(len(S))))
for i,row in enumerate(S):
    print('%4d ' % i + ''.join('%6.2f' % x for x in row))
print()
print('clone groups :', groups)
print('group sizes  :', {g: groups.count(g) for g in sorted(set(groups))})
print('SC           :', [round(x,3) for x in SC])

## 9. LP diagnostics with GLPK

The only solve the selector is allowed. Integrality is relaxed and GLPK gets 5 seconds.
LP-sol is the mean non-negative cosine between sorted 30-dimensional value profiles
(Eq. 15); LP-obj measures closeness to the pool median relaxed objective (Eq. 16).

In [ ]:
from ssc.sandbox import lp_relaxation
from ssc.lp_diag import lp_diagnostics, profile

t0 = time.time()
lp = [lp_relaxation(c) for c in codes]
print('relaxed %d candidates in %.1f s' % (len(lp), time.time()-t0))
print()
print('%-4s %-12s %14s %8s' % ('pos','status','relaxed obj','n vals'))
for i,r in enumerate(lp):
    print('%-4d %-12s %14s %8d' % (i, r['status'],
          round(r['objective'],4) if r.get('objective') is not None else '-',
          len(r.get('values') or [])))

lp_sol, lp_obj = lp_diagnostics(lp)
print()
print('LP-sol :', [round(float(x),3) for x in lp_sol])
print('LP-obj :', [round(float(x),3) for x in lp_obj])

feas = [i for i,r in enumerate(lp) if r['status']=='optimal' and r.get('objective') is not None]
if feas:
    z = sorted(lp[i]['objective'] for i in feas)
    m = z[len(z)//2]
    R = z[-1]-z[0] if z[-1]-z[0] >= 1e-9 else 1.0
    print()
    print('sorted relaxed objectives :', [round(x,3) for x in z])
    print('m (upper median) = %.4f   R = %.4f' % (m, R))

## 10. Fuse, select, then solve with SCIP

Eq. 8 forms the symbolic base, the fatal cap applies, Eq. 9 fuses in the two LP signals.
The argmax is the selected program. SCIP then solves every candidate's integer program
with a 120 second limit, and a candidate is correct when its objective matches gold within
the relative and absolute tolerances. The selector never sees these labels.

In [ ]:
from ssc.selector import variant_scores, VARIANTS
from ssc.sandbox import scip_solve
from ssc.evaluate import is_correct

scores = variant_scores(SC, SQ, TA, fatal, lp_sol, lp_obj)
B, SSC = scores['SSC_noLP'], scores['SSC']
pick   = max(range(len(SSC)), key=lambda i: SSC[i])

t0   = time.time()
scip = [scip_solve(c) for c in codes]
corr = [r['status']=='optimal' and r['objective'] is not None and is_correct(r['objective'], GOLD)
        for r in scip]
print('solved %d integer programs in %.1f s' % (len(scip), time.time()-t0))
print()
print('%-4s %7s %7s %7s %9s %8s %8s %9s %-11s %12s %8s'
      % ('pos','SC','SQ','TA','B','LP-sol','LP-obj','SSC','scip','objective','correct'))
for i in range(len(SSC)):
    print('%-4d %7.3f %7.3f %7.3f %9.4f %8.3f %8.3f %9.4f %-11s %12s %8s'
          % (i, SC[i], SQ[i], TA[i], B[i], lp_sol[i], lp_obj[i], SSC[i], scip[i]['status'],
             round(scip[i]['objective'],3) if scip[i]['objective'] is not None else '-', corr[i]))

print()
print('gold                  :', GOLD)
print('pool coverage         :', any(corr))
print('SSC picks position %d  : %s' % (pick, 'CORRECT' if corr[pick] else 'WRONG'))
print('first position        : %s' % ('CORRECT' if corr[0] else 'WRONG'))
print()
print('every selector variant on this pool')
for v in VARIANTS:
    sel = max(range(len(SSC)), key=lambda i: scores[v][i])
    print('  %-18s position %d  %s' % (v, sel, 'correct' if corr[sel] else 'wrong'))
print()
print('API usage for this problem :', llm.usage)